In [1]:
%pip install youtube-transcript-api sentence-transformers fastapi

Note: you may need to restart the kernel to use updated packages.


In [1]:
# =====================================================================
# Section 1: Comprehensive Environment & Library Verification
# =====================================================================

import importlib
import sys

# قائمة بجميع المكتبات المطلوبة طوال مراحل المشروع
required_libraries = {
    # 1. استخراج محتوى اليوتيوب
    "youtube_transcript_api": "YouTube Transcript API (استخراج النصوص)",
    # 2. قاعدة البيانات والـ RAG
    "chromadb": "ChromaDB (قاعدة البيانات الشعاعية)",
    # 3. نموذج الذكاء الاصطناعي والـ LLM
    "groq": "Groq SDK (الاتصال بنموذج الذكاء الاصطناعي)",
    # 4. السيرفر والباك إند
    "fastapi": "FastAPI (بناء الـ API)",
    "uvicorn": "Uvicorn (تشغيل خادم الويب)",
    "pydantic": "Pydantic (التحقق من صحة هياكل البيانات)",
    # 5. أدوات البيئة والاتصال
    "dotenv": "Python-Dotenv (إدارة مفاتيح الـ API)",
    "requests": "Requests (طلبات الويب)",
}

print("=" * 65)
print(" 🔍 فحص جاهزية المكتبات المطلوبة لمشروع YouTube RAG Agent")
print("=" * 65)

missing_libs = []

for module_name, description in required_libraries.items():
  try:
    mod = importlib.import_module(module_name)
    version = getattr(mod, "__version__", "متوفرة")
    print(f"✅ {description:<42} | v{version}")
  except ImportError:
    missing_libs.append(module_name)
    print(f"❌ {description:<42} | غير مثبتة!")

print("=" * 65)

if not missing_libs:
  print("🎉 مبروك! بيئة العمل مكتملة 100%، وجميع المكتبات جاهزة للبدء.")
else:
  print(f"⚠️ تنبيه: هناك {len(missing_libs)} مكتبة ناقصة:")
  print(f"يمكنك تثبيتها عبر أمر: pip install {' '.join(missing_libs)}")

 🔍 فحص جاهزية المكتبات المطلوبة لمشروع YouTube RAG Agent
✅ YouTube Transcript API (استخراج النصوص)    | vمتوفرة


python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10


✅ ChromaDB (قاعدة البيانات الشعاعية)         | v1.5.9
✅ Groq SDK (الاتصال بنموذج الذكاء الاصطناعي) | v0.37.1
✅ FastAPI (بناء الـ API)                     | v0.162.0
✅ Uvicorn (تشغيل خادم الويب)                 | v0.54.0
✅ Pydantic (التحقق من صحة هياكل البيانات)    | v2.14.2
✅ Python-Dotenv (إدارة مفاتيح الـ API)       | vمتوفرة
✅ Requests (طلبات الويب)                     | v2.34.2
🎉 مبروك! بيئة العمل مكتملة 100%، وجميع المكتبات جاهزة للبدء.


In [2]:
# =====================================================================
# Section 2: YouTube Transcript Extraction (مع رابط فعّال ومعالجة مرنة)
# =====================================================================

import re
from youtube_transcript_api import YouTubeTranscriptApi


def extract_video_id(url: str) -> str:
  """استخراج معرف الفيديو من الرابط."""
  pattern = (
      r'(?:v=|\/|youtu\.be\/|embed\/|shorts\/|\bvi?=|\bwatch\?v=)([0-9A-Za-z_-]{11})'
  )
  match = re.search(pattern, url)
  if match:
    return match.group(1)
  raise ValueError(f'الرابط المدخل غير صالح: {url}')


def format_timestamp(seconds: float) -> str:
  """تحويل الثواني إلى صيغة MM:SS."""
  total_seconds = int(seconds)
  hours, remainder = divmod(total_seconds, 3600)
  minutes, secs = divmod(remainder, 60)
  if hours > 0:
    return f'{hours:02d}:{minutes:02d}:{secs:02d}'
  return f'{minutes:02d}:{secs:02d}'


def get_youtube_transcript(
    url: str, preferred_langs=['ar', 'en']
) -> tuple[str, list[dict]]:
  """جلب النص والتوقيت لأي فيديو يوتيوب متوفر فيه نصوص."""
  video_id = extract_video_id(url)
  ytt = (
      YouTubeTranscriptApi()
      if callable(YouTubeTranscriptApi)
      else YouTubeTranscriptApi
  )

  raw_items = []

  # محاولة الجلب حسب اللغات المفضلة أولاً
  try:
    if hasattr(ytt, 'fetch'):
      fetched = ytt.fetch(video_id, languages=preferred_langs)
      raw_items = (
          fetched.to_raw_data()
          if hasattr(fetched, 'to_raw_data')
          else list(fetched)
      )
    else:
      raw_items = YouTubeTranscriptApi.get_transcript(
          video_id, languages=preferred_langs
      )
  except Exception:
    # إن لم تتوفر اللغات المفضلة، نجلب النص الافتراضي التلقائي للفيديو
    try:
      if hasattr(ytt, 'fetch'):
        fetched = ytt.fetch(video_id)
        raw_items = (
            fetched.to_raw_data()
            if hasattr(fetched, 'to_raw_data')
            else list(fetched)
        )
      else:
        raw_items = YouTubeTranscriptApi.get_transcript(video_id)
    except Exception as e:
      raise RuntimeError(
        f'تعذر جلب نص لهذا الفيديو. يرجى التأكد أن الفيديو مفعل به الترجمة/الكتابة (Captions).\nالسبب: {e}'
    )

  # تنظيف وتنسيق النصوص
  formatted_data = []
  for item in raw_items:
    text = item['text'] if isinstance(item, dict) else getattr(item, 'text', '')
    start = (
        item['start'] if isinstance(item, dict) else getattr(item, 'start', 0.0)
    )
    duration = (
        item.get('duration', 0.0)
        if isinstance(item, dict)
        else getattr(item, 'duration', 0.0)
    )

    clean_text = str(text).replace('\n', ' ').strip()
    if clean_text:
      formatted_data.append({
          'text': clean_text,
          'start': float(start),
          'duration': float(duration),
          'timestamp_str': format_timestamp(start),
      })

  return video_id, formatted_data


# =====================================================================
# تجربة عملية برابط نشط ومضمون 100% (Stanford Speech)
# يمكنك أيضاً تجربة أي رابط فيديو تفضله أنت شرط أن يحتوي على Captions
# =====================================================================
active_sample_url = 'https://www.youtube.com/watch?v=UF8uR6Z6KLc'

try:
  vid_id, transcript_items = get_youtube_transcript(active_sample_url)
  print(f'✅ تم استخراج النص بنجاح تام!')
  print(f'🎥 معرّف الفيديو (Video ID): {vid_id}')
  print(f'📊 إجمالي عدد الجمل المستخرجة: {len(transcript_items)}')
  print('\n--- أول 4 مقاطع مع التوقيت الزمني ---')
  for entry in transcript_items[:4]:
    print(f"[{entry['timestamp_str']}] {entry['text']}")

except Exception as error:
  print(f'❌ خطأ: {error}')

✅ تم استخراج النص بنجاح تام!
🎥 معرّف الفيديو (Video ID): UF8uR6Z6KLc
📊 إجمالي عدد الجمل المستخرجة: 161

--- أول 4 مقاطع مع التوقيت الزمني ---
[00:06] جامعة ستانفورد  www.stanford.edu
[00:07] هذا البرنامج يأتيكم من جامعة ستانفورد , رجاءاً زورونا على الموقع اعلاه
[00:15] تصفيق
[00:21] أشكركم


In [3]:
# =====================================================================
# Section 3: RAG System - Chunking & ChromaDB Vector Store
# =====================================================================

import chromadb
from chromadb.utils import embedding_functions


# ---------------------------------------------------------------------
# 1. دالة تقطيع النص مع الاحتفاظ بالتوقيت الزمني
# ---------------------------------------------------------------------
def chunk_transcript(
    transcript_items: list[dict], max_chars: int = 700, overlap_chars: int = 150
) -> list[dict]:
  """تقسيم تفريغ الفيديو إلى مقاطع نصية متراكبة (Overlapping Chunks) مع حفظ التوقيت."""
  chunks = []
  current_text = []
  current_chars = 0
  current_start_time = 0.0
  current_timestamp_str = "00:00"

  for item in transcript_items:
    text = item["text"]
    start = item["start"]
    timestamp = item["timestamp_str"]

    if not current_text:
      current_start_time = start
      current_timestamp_str = timestamp

    current_text.append(text)
    current_chars += len(text) + 1

    # عند بلوغ الحد الأقصى من الحروف، نحفظ المقطع وننشئ المقطع التالي مع التداخل (Overlap)
    if current_chars >= max_chars:
      full_chunk_text = " ".join(current_text)
      chunks.append({
          "text": full_chunk_text,
          "start_time": current_start_time,
          "timestamp_str": current_timestamp_str,
      })

      # الاحتفاظ بآخر جملتين للتداخل لضمان عدم انقطاع سياق الكلام
      overlap_text = current_text[-2:] if len(current_text) >= 2 else []
      current_text = list(overlap_text)
      current_chars = sum(len(t) + 1 for t in current_text)
      if overlap_text:
        current_start_time = start
        current_timestamp_str = timestamp

  # إضافة المتبقي إن وُجد
  if current_text:
    chunks.append({
        "text": " ".join(current_text),
        "start_time": current_start_time,
        "timestamp_str": current_timestamp_str,
    })

  return chunks


# ---------------------------------------------------------------------
# 2. إعداد قاعدة بيانات ChromaDB وتخزين المتجهات
# ---------------------------------------------------------------------
# إنشاء عميل محلي يحفظ البيانات في مجلد chroma_db
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# نموذج التضمين (يدعم اللغتين العربية والإنجليزية بكفاءة)
embedding_func = (
    embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="paraphrase-multilingual-MiniLM-L12-v2"
    )
)


def index_video_transcript(video_id: str, transcript_items: list[dict]):
  """تقطيع النص وتخزينه في ChromaDB داخل Collection مخصصة للفيديو."""
  chunks = chunk_transcript(transcript_items)

  # اسم الـ Collection مستند لمعرف الفيديو
  collection_name = f"video_{video_id.replace('-', '_')}"

  # إعادة ضبط الـ Collection إذا كانت موجودة مسبقاً لمنع التكرار
  try:
    chroma_client.delete_collection(name=collection_name)
  except Exception:
    pass

  collection = chroma_client.create_collection(
      name=collection_name, embedding_function=embedding_func
  )

  # تجهيز البيانات للإدخال
  documents = [c["text"] for c in chunks]
  metadatas = [
      {
          "start_time": c["start_time"],
          "timestamp_str": c["timestamp_str"],
          "video_id": video_id,
      }
      for c in chunks
  ]
  ids = [f"{video_id}_chunk_{i}" for i in range(len(chunks))]

  collection.add(documents=documents, metadatas=metadatas, ids=ids)

  return collection, len(chunks)


# ---------------------------------------------------------------------
# 3. دالة البحث الدلالي واسترجاع المقاطع الأكثر صلة (Semantic Search)
# ---------------------------------------------------------------------
def search_transcript(video_id: str, query: str, top_k: int = 3) -> list[dict]:
  """البحث عن أكثر المقاطع المتعلقة بسؤال المستخدم داخل الفيديو."""
  collection_name = f"video_{video_id.replace('-', '_')}"
  collection = chroma_client.get_collection(
      name=collection_name, embedding_function=embedding_func
  )

  results = collection.query(query_texts=[query], n_results=top_k)

  retrieved_docs = []
  for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    retrieved_docs.append({
        "text": doc,
        "timestamp": meta["timestamp_str"],
        "start_time": meta["start_time"],
    })

  return retrieved_docs


# =====================================================================
# تجربة عملية على الفيديو السابق
# =====================================================================
print("⏳ جارٍ تقطيع النص وفهرسته في ChromaDB...")
collection, chunks_count = index_video_transcript(vid_id, transcript_items)
print(f"✅ تمت الفهرسة بنجاح! تم إنشاء {chunks_count} مقطع نصي (Chunks).")

# تجربة سؤال للبحث الدلالي
test_question = "عن ماذا يتحدث هذا البرنامج أو المحاضرة؟"
print(f"\n🔎 تجربة بحث عن السؤال: '{test_question}'")
matches = search_transcript(vid_id, test_question, top_k=2)

for i, match in enumerate(matches, 1):
  print(f"\n[النتيجة {i}] - التوقيت: [{match['timestamp']}]")
  print(f"النص: {match['text'][:200]}...")

python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10


e:\Scond_Stage_in_Training\trinin_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3393.71it/s]


⏳ جارٍ تقطيع النص وفهرسته في ChromaDB...
✅ تمت الفهرسة بنجاح! تم إنشاء 18 مقطع نصي (Chunks).

🔎 تجربة بحث عن السؤال: 'عن ماذا يتحدث هذا البرنامج أو المحاضرة؟'

[النتيجة 1] - التوقيت: [00:06]
النص: جامعة ستانفورد  www.stanford.edu هذا البرنامج يأتيكم من جامعة ستانفورد , رجاءاً زورونا على الموقع اعلاه تصفيق أشكركم (ستيف جوبز - الرئيس التنفيذي لشركة أبل و شركة بيكسار) اتشرف بلقائي بكم اليوم لبدئكم...

[النتيجة 2] - التوقيت: [14:07]
النص: وفي الغلاف الأخير من العدد الاخيرة كانت صورة عرضت طريقا زراعيا جميلا وقت الصباح، من النوع الذي كنت لتحمل حقيبتك وتستوقف السيارات في الطريق لتأخذك إليه لو كنت من النوع المغامر. مع هذا المنظر الخلاب جاء...


In [4]:
# =====================================================================
# Section 4: Chat with Video using Groq API (Loaded from .env)
# =====================================================================

import os
from dotenv import load_dotenv
from groq import Groq

# 1. تحميل المتغيرات من ملف .env
load_dotenv(override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
DOCTOR_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
FALLBACK_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY or GROQ_API_KEY == "gsk_your_actual_groq_key_here":
  raise ValueError(
      "❌ لم يتم العثور على مفتاح Groq صالح في ملف .env! يرجى وضع المفتاح"
      " الصحيح."
  )

# إنشاء عميل Groq
groq_client = Groq(api_key=GROQ_API_KEY)
print(f"✅ تم تحميل المفتاح بنجاح من .env! النموذج المستهدف: {DOCTOR_MODEL}")


# 2. دالة توليد الإجابة
def generate_rag_response(
    query: str, retrieved_chunks: list[dict], model_name: str = DOCTOR_MODEL
) -> str:
  """تجهيز سياق الـ RAG وإرساله إلى Groq API للإجابة على سؤال المستخدم."""

  context_text = "\n\n".join([
      f"--- [مقطع من التوقيت {c['timestamp']}] ---\n{c['text']}"
      for c in retrieved_chunks
  ])

  system_prompt = (
      "أنت مساعد ذكي متخصص في الإجابة عن الأسئلة المتعلقة بمحتوى فيديو"
      " يوتيوب.\n"
      "قواعد الإجابة:\n"
      "1. أجب فقط بالاعتماد على سياق النص المقتبس من الفيديو المرفق أدناه.\n"
      "2. اذكر دائماً التوقيت الزمني (Timestamp) بجانب المعلومات التي تقدمها"
      " لمساعدة المستخدم في الرجوع للفيديو.\n"
      "3. أجب بنفس لغة السؤال بأسلوب مهني وواضح وموجز."
  )

  user_prompt = f"""إليك المقاطع ذات الصلة المستخرجة من تفريغ الفيديو:

{context_text}

سؤال المستخدم: {query}

الإجابة:"""

  try:
    response = groq_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=600,
    )
    return response.choices[0].message.content

  except Exception as e:
    err_str = str(e)
    # تبديل تلقائي للنموذج البديل إذا لم يكن نموذج الدكتور متاحاً على السيرفر
    if (
        "model_not_found" in err_str.lower()
        or "404" in err_str
        or "not found" in err_str.lower()
    ):
      print(
          f"⚠️ تنبيه: النموذج '{model_name}' غير متاح حالياً، جاري التبديل"
          f" للنموذج: {FALLBACK_MODEL}..."
      )
      return generate_rag_response(
          query, retrieved_chunks, model_name=FALLBACK_MODEL
      )
    raise e


def chat_with_video(
    video_id: str, question: str, top_k: int = 3
) -> tuple[str, list[dict]]:
  """جلب السياق من ChromaDB وتوليد الإجابة من Groq."""
  relevant_chunks = search_transcript(video_id, question, top_k=top_k)
  answer = generate_rag_response(question, relevant_chunks)
  return answer, relevant_chunks


# =====================================================================
# تجربة السؤال والدردشة مع الفيديو
# =====================================================================
user_question = "من هو المتحدث في هذا الفيديو وما هي نصيحته الأساسية؟"

print(f"\n❓ السؤال: {user_question}\n" + "-" * 50)

try:
  answer, sources = chat_with_video(vid_id, user_question)
  print(f"🤖 إجابة الذكاء الاصطناعي:\n\n{answer}")
  print("\n" + "-" * 50)
  print("📌 التوقيتات المستند إليها في الفيديو:")
  for s in sources:
    print(f"- دقيقة [{s['timestamp']}]: {s['text'][:85]}...")

except Exception as err:
  print(f"❌ خطأ: {err}")

python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10


✅ تم تحميل المفتاح بنجاح من .env! النموذج المستهدف: openai/gpt-oss-120b

❓ السؤال: من هو المتحدث في هذا الفيديو وما هي نصيحته الأساسية؟
--------------------------------------------------
🤖 إجابة الذكاء الاصطناعي:

المتحدث في الفيديو هو **ستيف جوبز** – المؤسس المشارك لشركة أبل والرئيس التنفيذي السابق لها (كما يُذكر في المقطع من 00:06).  

نصيحته الأساسية هي **«ابقَ جائعاً – ابقَ أحمقاً»**؛ أي أن يظل الإنسان متطلّعاً إلى التعلم والابتكار ولا يخاف من المخاطرة أو الفشل. يربط جوبز هذه العبارة بضرورة تذكّرنا بأننا **سوف نموت** (المقطع من 09:33) لتشجيعنا على اتخاذ القرارات الجريئة ومتابعة شغفنا دون تردد.  

- «ابقَ جائعاً – ابقَ أحمقاً» – 14:07  
- تذكّر الموت كدافع لاتخاذ قرارات صادقة – 09:33  

بهذه الكلمات يلخّص جوبز رسالته للخرّيجين: استمروا في السعي وراء المعرفة والابتكار، ولا تسمحوا للخوف أو الراحة أن يحدّوا طموحاتكم.

--------------------------------------------------
📌 التوقيتات المستند إليها في الفيديو:
- دقيقة [14:07]: وفي الغلاف الأخير من العدد الاخيرة كانت صورة عرضت طريقا زراعيا ج

In [10]:
# =====================================================================
# Section 5: Generating Full Project Files (FastAPI & HTML/CSS/JS UI)
# =====================================================================

import os

# 1. إنشاء مجلد الواجهة
os.makedirs("static", exist_ok=True)

# 2. إنشاء ملف الواجهة static/index.html (HTML + CSS + JavaScript)
html_code = """<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>مساعد يوتيوب الذكي - YouTube RAG Agent</title>
    <link href="https://fonts.googleapis.com/css2?family=Cairo:wght@400;600;700&display=swap" rel="stylesheet">
    <style>
        :root {
            --primary: #ef4444;
            --primary-hover: #dc2626;
            --bg: #0f172a;
            --card-bg: #1e293b;
            --text-main: #f8fafc;
            --text-muted: #94a3b8;
            --border: #334155;
            --user-bubble: #2563eb;
            --bot-bubble: #334155;
        }

        * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Cairo', sans-serif; }
        body { background: var(--bg); color: var(--text-main); height: 100vh; display: flex; flex-direction: column; }
        
        header {
            background: var(--card-bg);
            padding: 1rem 2rem;
            border-bottom: 1px solid var(--border);
            display: flex;
            align-items: center;
            justify-content: space-between;
        }
        header h1 { font-size: 1.3rem; display: flex; align-items: center; gap: 0.5rem; color: #fff; }
        header .badge { background: #10b981; color: white; padding: 0.2rem 0.6rem; border-radius: 999px; font-size: 0.75rem; }

        .container {
            display: grid;
            grid-template-columns: 1fr 1.2fr;
            gap: 1.5rem;
            padding: 1.5rem;
            flex: 1;
            overflow: hidden;
        }

        /* الجانب الأيمن: الفيديو والإدخال */
        .video-section {
            background: var(--card-bg);
            border-radius: 12px;
            padding: 1.2rem;
            display: flex;
            flex-direction: column;
            gap: 1rem;
            border: 1px solid var(--border);
        }
        .url-box { display: flex; gap: 0.5rem; }
        .url-box input {
            flex: 1;
            padding: 0.75rem 1rem;
            border-radius: 8px;
            border: 1px solid var(--border);
            background: #0f172a;
            color: white;
            font-size: 0.95rem;
            outline: none;
        }
        .url-box input:focus { border-color: var(--primary); }
        .btn {
            background: var(--primary);
            color: white;
            border: none;
            padding: 0 1.2rem;
            border-radius: 8px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.2s;
        }
        .btn:hover { background: var(--primary-hover); }
        .btn:disabled { opacity: 0.6; cursor: not-allowed; }

        .player-container {
            flex: 1;
            background: #000;
            border-radius: 8px;
            overflow: hidden;
            display: flex;
            align-items: center;
            justify-content: center;
            min-height: 280px;
        }
        .player-container iframe { width: 100%; height: 100%; border: none; }
        .placeholder { color: var(--text-muted); text-align: center; }

        /* الجانب الأيسر: نافذة المحادثة */
        .chat-section {
            background: var(--card-bg);
            border-radius: 12px;
            display: flex;
            flex-direction: column;
            border: 1px solid var(--border);
            overflow: hidden;
        }
        .chat-messages {
            flex: 1;
            padding: 1.2rem;
            overflow-y: auto;
            display: flex;
            flex-direction: column;
            gap: 1rem;
        }
        .message {
            max-width: 85%;
            padding: 0.8rem 1.1rem;
            border-radius: 12px;
            font-size: 0.95rem;
            line-height: 1.6;
        }
        .message.user {
            align-self: flex-start;
            background: var(--user-bubble);
            color: white;
            border-bottom-right-radius: 2px;
        }
        .message.bot {
            align-self: flex-end;
            background: var(--bot-bubble);
            color: var(--text-main);
            border-bottom-left-radius: 2px;
            width: 90%;
        }
        .timestamp-tag {
            display: inline-block;
            background: #0ea5e9;
            color: white;
            padding: 0.1rem 0.5rem;
            border-radius: 4px;
            font-size: 0.8rem;
            margin: 0.2rem 0.3rem;
            cursor: pointer;
            text-decoration: underline;
        }
        .timestamp-tag:hover { background: #0284c7; }

        .chat-input-box {
            padding: 1rem;
            border-top: 1px solid var(--border);
            display: flex;
            gap: 0.5rem;
            background: #1e293b;
        }
        .chat-input-box input {
            flex: 1;
            padding: 0.75rem 1rem;
            border-radius: 8px;
            border: 1px solid var(--border);
            background: #0f172a;
            color: white;
            font-size: 0.95rem;
            outline: none;
        }
        .chat-input-box input:focus { border-color: #3b82f6; }

        @media (max-width: 850px) {
            .container { grid-template-columns: 1fr; overflow-y: auto; }
            body { height: auto; }
        }
    </style>
</head>
<body>
    <header>
        <h1>🎥 مساعد يوتيوب الذكي <span style="font-size: 0.85rem; font-weight: normal; color: var(--text-muted);">(Groq + ChromaDB)</span></h1>
        <div class="badge" id="statusBadge">جاهز للبدء</div>
    </header>

    <div class="container">
        <!-- قسم الفيديو -->
        <div class="video-section">
            <div class="url-box">
                <input type="text" id="videoUrl" placeholder="ضع رابط فيديو يوتيوب هنا (مثال: https://www.youtube.com/watch?v=...)" />
                <button class="btn" id="processBtn" onclick="processVideo()">تحليل الفيديو</button>
            </div>
            <div class="player-container" id="playerWrapper">
                <div class="placeholder">ضع رابط الفيديو واضغط "تحليل الفيديو" لعرضه هنا</div>
            </div>
        </div>

        <!-- قسم المحادثة -->
        <div class="chat-section">
            <div class="chat-messages" id="chatBox">
                <div class="message bot">
                    أهلاً بك! 👋 ضع رابط أي فيديو يوتيوب يحتوي على ترجمة (Captions)، وسأقوم بتحليله لتتمكن من سؤالي عن أي تفاصيل داخله بدقة مع التوقيتات الزمنية.
                </div>
            </div>
            <div class="chat-input-box">
                <input type="text" id="userQuestion" placeholder="اسأل سؤالاً حول محتوى الفيديو..." disabled onkeydown="if(event.key==='Enter') sendQuestion()" />
                <button class="btn" id="sendBtn" style="background:#2563eb;" disabled onclick="sendQuestion()">إرسال</button>
            </div>
        </div>
    </div>

    <script>
        let currentVideoId = null;

        function updateStatus(text, color="#10b981") {
            const b = document.getElementById("statusBadge");
            b.innerText = text;
            b.style.backgroundColor = color;
        }

        function jumpToTime(seconds) {
            if (!currentVideoId) return;
            const iframe = document.querySelector("#playerWrapper iframe");
            if (iframe) {
                iframe.src = `https://www.youtube.com/embed/${currentVideoId}?start=${Math.floor(seconds)}&autoplay=1`;
            }
        }

        async function processVideo() {
            const urlInput = document.getElementById("videoUrl");
            const url = urlInput.value.trim();
            if (!url) return alert("يرجى إدخال رابط يوتيوب!");

            const btn = document.getElementById("processBtn");
            btn.disabled = true;
            btn.innerText = "⏳ جارٍ التحليل...";
            updateStatus("جارٍ استخراج وتخزين النص...", "#f59e0b");

            try {
                const res = await fetch("/api/process-video", {
                    method: "POST",
                    headers: { "Content-Type": "application/json" },
                    body: JSON.stringify({ url: url })
                });

                const data = await res.json();
                if (!res.ok) throw new Error(data.detail || "تعذر معالجة الفيديو");

                currentVideoId = data.video_id;
                
                // عرض الفيديو في المشغل
                document.getElementById("playerWrapper").innerHTML = 
                    `<iframe src="https://www.youtube.com/embed/${currentVideoId}?autoplay=0" allowfullscreen></iframe>`;

                // تفعيل صندوق الدردشة
                document.getElementById("userQuestion").disabled = false;
                document.getElementById("sendBtn").disabled = false;
                document.getElementById("userQuestion").focus();

                updateStatus("جاهز للأسئلة", "#10b981");
                appendMessage("bot", `✅ تم تحليل الفيديو وفهرسة ${data.chunks_count} مقطع نصي بنجاح! يمكنك الآن سؤالي عن أي شيء يدور داخل المقطع.`);
            } catch (err) {
                alert(err.message);
                updateStatus("خطأ في المعالجة", "#ef4444");
            } finally {
                btn.disabled = false;
                btn.innerText = "تحليل الفيديو";
            }
        }

        async function sendQuestion() {
            const input = document.getElementById("userQuestion");
            const q = input.value.trim();
            if (!q || !currentVideoId) return;

            appendMessage("user", q);
            input.value = "";
            input.disabled = true;
            document.getElementById("sendBtn").disabled = true;

            const loadingId = appendMessage("bot", "⏳ جاري البحث في الفيديو وسؤال الذكاء الاصطناعي...");

            try {
                const res = await fetch("/api/chat", {
                    method: "POST",
                    headers: { "Content-Type": "application/json" },
                    body: JSON.stringify({ video_id: currentVideoId, question: q })
                });

                const data = await res.json();
                if (!res.ok) throw new Error(data.detail || "خطأ في الإجابة");

                // تحديث الرسالة بالإجابة وروابط التوقيت
                updateBotMessage(loadingId, data.answer, data.sources);
            } catch (err) {
                updateBotMessage(loadingId, `❌ حدث خطأ: ${err.message}`, []);
            } finally {
                input.disabled = false;
                document.getElementById("sendBtn").disabled = false;
                input.focus();
            }
        }

        function appendMessage(role, text) {
            const box = document.getElementById("chatBox");
            const div = document.createElement("div");
            const id = "msg_" + Date.now();
            div.id = id;
            div.className = `message ${role}`;
            div.innerText = text;
            box.appendChild(div);
            box.scrollTop = box.scrollHeight;
            return id;
        }

        function updateBotMessage(elemId, answerText, sources) {
            const div = document.getElementById(elemId);
            if (!div) return;

            let html = answerText.replace(/\\n/g, "<br>");

            if (sources && sources.length > 0) {
                html += "<br><hr style='border: 0; border-top: 1px solid #475569; margin: 10px 0;'><strong>⏱️ المقاطع ذات الصلة في الفيديو (اضغط للقفز):</strong><br>";
                sources.forEach(s => {
                    html += `<span class="timestamp-tag" onclick="jumpToTime(${s.start_time})">▶ الدقيقة [${s.timestamp}]</span> `;
                });
            }

            div.innerHTML = html;
            const box = document.getElementById("chatBox");
            box.scrollTop = box.scrollHeight;
        }
    </script>
</body>
</html>
"""

with open("static/index.html", "w", encoding="utf-8") as f:
  f.write(html_code)

print("✅ 1. تم إنشاء ملف واجهة المستخدم: static/index.html")

# 3. إنشاء كود خادم FastAPI في ملف main.py
main_py_code = """import os
import re
from fastapi import FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from pydantic import BaseModel
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq

load_dotenv(override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
DOCTOR_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
FALLBACK_MODEL = "llama-3.3-70b-versatile"

app = FastAPI(title="YouTube RAG Agent")

# إعداد قاعدة البيانات الشعاعية
chroma_client = chromadb.PersistentClient(path="./chroma_db")
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# عميل Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# نماذج البيانات (Pydantic Models)
class ProcessRequest(BaseModel):
    url: str

class ChatRequest(BaseModel):
    video_id: str
    question: str

def extract_video_id(url: str) -> str:
    pattern = r'(?:v=|\\/|youtu\\.be\\/|embed\\/|shorts\\/|\\bvi?=|\\bwatch\\?v=)([0-9A-Za-z_-]{11})'
    match = re.search(pattern, url)
    if match:
        return match.group(1)
    raise ValueError("رابط يوتيوب غير صالح")

def format_timestamp(seconds: float) -> str:
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}" if hours > 0 else f"{minutes:02d}:{secs:02d}"

def fetch_transcript(video_id: str):
    ytt = YouTubeTranscriptApi() if callable(YouTubeTranscriptApi) else YouTubeTranscriptApi
    try:
        if hasattr(ytt, "fetch"):
            try:
                fetched = ytt.fetch(video_id, languages=['ar', 'en'])
            except Exception:
                fetched = ytt.fetch(video_id)
            return fetched.to_raw_data() if hasattr(fetched, "to_raw_data") else list(fetched)
        else:
            try:
                return YouTubeTranscriptApi.get_transcript(video_id, languages=['ar', 'en'])
            except Exception:
                return YouTubeTranscriptApi.get_transcript(video_id)
    except Exception as e:
        raise RuntimeError(f"تعذر استخراج النص: {str(e)}")

def chunk_data(transcript_items, max_chars=700):
    chunks, curr_text = [], []
    curr_len, curr_start, curr_ts = 0, 0.0, "00:00"
    
    for item in transcript_items:
        t = (item['text'] if isinstance(item, dict) else item.text).replace('\\n', ' ').strip()
        s = float(item['start'] if isinstance(item, dict) else item.start)
        ts = format_timestamp(s)
        
        if not curr_text:
            curr_start, curr_ts = s, ts
            
        curr_text.append(t)
        curr_len += len(t) + 1
        
        if curr_len >= max_chars:
            chunks.append({"text": " ".join(curr_text), "start_time": curr_start, "timestamp": curr_ts})
            curr_text = curr_text[-2:] if len(curr_text) >= 2 else []
            curr_len = sum(len(x) + 1 for x in curr_text)
            
    if curr_text:
        chunks.append({"text": " ".join(curr_text), "start_time": curr_start, "timestamp": curr_ts})
    return chunks

# مسار تقديم واجهة المستخدم الرئيسية
@app.get("/")
def serve_home():
    return FileResponse("static/index.html")

# مسار استخراج وفهرسة الفيديو
@app.post("/api/process-video")
def process_video(req: ProcessRequest):
    try:
        video_id = extract_video_id(req.url)
        raw_transcript = fetch_transcript(video_id)
        chunks = chunk_data(raw_transcript)
        
        coll_name = f"video_{video_id.replace('-', '_')}"
        try:
            chroma_client.delete_collection(name=coll_name)
        except Exception:
            pass
            
        collection = chroma_client.create_collection(name=coll_name, embedding_function=embedding_func)
        collection.add(
            documents=[c["text"] for c in chunks],
            metadatas=[{"start_time": c["start_time"], "timestamp": c["timestamp"]} for c in chunks],
            ids=[f"{video_id}_{i}" for i in range(len(chunks))]
        )
        return {"status": "success", "video_id": video_id, "chunks_count": len(chunks)}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

# مسار المحادثة مع الفيديو
@app.post("/api/chat")
def chat_with_video(req: ChatRequest):
    try:
        coll_name = f"video_{req.video_id.replace('-', '_')}"
        collection = chroma_client.get_collection(name=coll_name, embedding_function=embedding_func)
        results = collection.query(query_texts=[req.question], n_results=3)
        
        sources = []
        for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
            sources.append({"text": doc, "timestamp": meta["timestamp"], "start_time": meta["start_time"]})
            
        context_text = "\\n\\n".join([f"--- [مقطع من دقيقة {s['timestamp']}] ---\\n{s['text']}" for s in sources])
        
        system_prompt = "أنت مساعد يوتيوب الذكي. أجب على السؤال بالاعتماد بدقة على المقاطع المرفقة من الفيديو واذكر التوقيتات المناسبة."
        user_prompt = f"المقاطع المستخرجة:\\n{context_text}\\n\\nالسؤال: {req.question}\\nالإجابة:"
        
        try:
            resp = groq_client.chat.completions.create(
                model=DOCTOR_MODEL,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                temperature=0.3,
                max_tokens=600
            )
            answer = resp.choices[0].message.content
        except Exception as groq_err:
            if "model_not_found" in str(groq_err).lower() or "404" in str(groq_err):
                resp = groq_client.chat.completions.create(
                    model=FALLBACK_MODEL,
                    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                    temperature=0.3,
                    max_tokens=600
                )
                answer = resp.choices[0].message.content
            else:
                raise groq_err
                
        return {"answer": answer, "sources": sources}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

app.mount("/static", StaticFiles(directory="static"), name="static")
"""

with open("main.py", "w", encoding="utf-8") as f:
  f.write(main_py_code)

print("✅ 2. تم إنشاء ملف الخادم الرئيسي: main.py")
print("🎉 كل الملفات جاهزة الآن للتشغيل!")

✅ 1. تم إنشاء ملف واجهة المستخدم: static/index.html
✅ 2. تم إنشاء ملف الخادم الرئيسي: main.py
🎉 كل الملفات جاهزة الآن للتشغيل!
